In [16]:
# Used to securely store your API key
# from google.colab import userdata
 
# Retrieve the Groq API key from Colab secrets

 
# Ensure the .env file exists and write the key to it
# with open('.env', 'a') as f:
#     f.write(f'GROQ_API_KEY="{GROQ_API_KEY}"\n')
 
# print("GROQ_API_KEY written to .env file.")
 
import os, json
from dotenv import load_dotenv
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq
 
 
load_dotenv()
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.1-8b-instant",
    groq_api_key=GROQ_API_KEY
)
 
# --- shared state schema ---
class AgentState(TypedDict):
    goal:       str
    tasks:      List[str]
    results:    List[str]
    critique:   str
    approved:   bool
    iterations: int
 
def planner(state: AgentState) -> AgentState:
    system = """You are a planning agent. Break the user's goal into
at most 5 concrete, actionable tasks. Respond ONLY with a
valid JSON array of strings. No preamble, no markdown."""
 
    messages = [
        SystemMessage(content=system),
        HumanMessage(content=f"Goal: {state['goal']}")
    ]
    response = llm.invoke(messages).content.strip()
 
    try:
        clean = response.replace("```json","").replace("```","").strip()
        tasks = json.loads(clean)
    except json.JSONDecodeError:
        tasks = [response]   # fallback: treat whole response as one task
 
    print(f"\n[Planner] Generated {len(tasks)} tasks:")
    for i, t in enumerate(tasks): print(f"  {i+1}. {t}")
 
    return {**state, "tasks": tasks}
 
initial_state: AgentState = {
    "goal":       "Research and summarise the top 3 trends in agriculture for 2025",
    "tasks":      [],
    "results":    [],
    "critique":   "",
    "approved":   False,
    "iterations": 0
}
planner(initial_state)
 



[Planner] Generated 5 tasks:
  1. Identify credible sources of agricultural research and trends
  2. Research and gather information on the top 3 trends in agriculture for 2025
  3. Analyze and categorize the gathered information into relevant trends
  4. Evaluate and prioritize the top 3 trends based on relevance and impact
  5. Create a concise summary of the top 3 trends in agriculture for 2025


{'goal': 'Research and summarise the top 3 trends in agriculture for 2025',
 'tasks': ['Identify credible sources of agricultural research and trends',
  'Research and gather information on the top 3 trends in agriculture for 2025',
  'Analyze and categorize the gathered information into relevant trends',
  'Evaluate and prioritize the top 3 trends based on relevance and impact',
  'Create a concise summary of the top 3 trends in agriculture for 2025'],
 'results': [],
 'critique': '',
 'approved': False,
 'iterations': 0}